# Excitable FitzHugh-Nagumo with a parametric FNO

This takes the oscillatory FHN operator-learning setup into the excitable regime: a single stable rest state, where only a super-threshold bump fires a travelling action potential. Same semi-implicit solver and FiLM-conditioned FNO as the main paper, but on a longer domain (L=8) so pulses stay localized instead of filling the box.

We show three things: all-or-none firing, travelling pulses with conduction velocity scaling as sqrt(Du), and single-step plus rollout accuracy.

Runtime > Change runtime type > GPU. End to end takes about 20-40 min on a T4/L4.

## 0. Setup

In [ ]:
import math, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print("gpu   :", torch.cuda.get_device_name(0))

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if device == "cuda":
    torch.cuda.manual_seed_all(SEED)


## Configuration

The ranges below sit in the excitable regime; the next section checks that and resamples any bad draws. We use a longer domain (L=8) than the oscillatory study so the pulse stays localized, and a large recovery time tau so the refractory tail is visible within T=40.

In [ ]:
NX        = 256
L         = 8.0             # longer domain than the oscillatory study
DX        = L / NX
DT        = 0.02
T         = 40.0
N_SAVE    = 100             # 101 frames including the IC
DT_SAVE   = (int(T / DT) // N_SAVE) * DT

# lambda = (Du, Dv, a, b, tau)
DU_RANGE  = (0.006, 0.030)
DV_RANGE  = (0.0005, 0.005)
A_RANGE   = (0.70, 0.95)
B_RANGE   = (0.60, 1.00)
TAU_RANGE = (8.0, 20.0)

# Gaussian bump on the rest state, physical units x in [0, L].
# It must be wide enough to launch a pulse: too narrow and diffusion
# smears it below threshold before it organizes.
AMP_RANGE   = (0.20, 2.50)   # spans sub- and super-threshold
SIGMA_RANGE = (0.30, 0.55)
X0_RANGE    = (0.30 * L, 0.70 * L)

N_TRAIN   = 1000
N_VAL     = 250
GEN_CHUNK = 100

# model
MODES     = 16
WIDTH     = 64
N_LAYERS  = 6
IN_CH     = 2
OUT_CH    = 2
N_PARAMS  = 5

# training
EPOCHS    = 40
BATCH     = 64
LR        = 1e-3
WD        = 1e-4
GRAD_CLIP = 1.0

print(f"DT_SAVE = {DT_SAVE:.3f}  ->  {N_SAVE} steps span T = {N_SAVE*DT_SAVE:.1f}")


## Fixed point, excitability test, GPU solver

Setting the time derivatives to zero gives the cubic u^3 + (3/b - 3)u + 3a/b = 0, with v* = (u*+a)/b. For these ranges it has a single real root with u* < -1, so the medium is monostable. The rest state is stable (hence excitable) when the reaction Jacobian has tr J < 0 and det J > 0, which we check below.

In [ ]:
def rest_state(a, b):
    """Left root of the rest-state cubic, and v* = (u*+a)/b."""
    roots = np.roots([1.0, 0.0, (3.0 / b - 3.0), 3.0 * a / b])
    real = roots[np.abs(roots.imag) < 1e-9].real
    u_star = float(np.min(real))        # left branch: u* < -1
    v_star = (u_star + a) / b
    return u_star, v_star

def is_excitable(a, b, tau):
    u_star, _ = rest_state(a, b)
    tr  = (1.0 - u_star**2) - b / tau
    det = (1.0 - b * (1.0 - u_star**2)) / tau
    return (tr < 0.0) and (det > 0.0), u_star

# check the corners of the parameter box
for a in A_RANGE:
    for b in B_RANGE:
        for tau in TAU_RANGE:
            ok, us = is_excitable(a, b, tau)
            print(f"a={a:.2f} b={b:.2f} tau={tau:4.1f} -> u*={us:+.3f}  excitable={ok}")


In [ ]:
class FHNSolverGPU:
    """Batched semi-implicit FHN solver, matches FDBackendGPU.solve_batched in the repo."""

    def __init__(self, nx=NX, dx=DX, device=device, dtype=torch.float32):
        self.nx, self.dx, self.device, self.dtype = nx, dx, device, dtype
        self.lap = self._laplacian_1d(nx, dx).to(device=device, dtype=dtype)
        self.I = torch.eye(nx, device=device, dtype=dtype)

    def _laplacian_1d(self, n, dx):
        lap = torch.zeros((n, n), dtype=torch.float64)
        c = 1.0 / (dx * dx)
        idx = torch.arange(n)
        lap[idx, idx] = -2 * c
        lap[idx, (idx + 1) % n] = c
        lap[idx, (idx - 1) % n] = c
        return lap

    @torch.no_grad()
    def solve_batched(self, u0, v0, params, T=T, dt=DT, n_save=N_SAVE, I_ext=None):
        """u0,v0:(B,nx)  params:(B,5)=[Du,Dv,a,b,tau]  I_ext:(B,nx) or None.
        Returns u,v:(B,n_save+1,nx) and t:(n_save+1,)."""
        B, nx = u0.shape
        n_steps = int(T / dt)
        save_interval = max(1, n_steps // n_save)

        p = params.to(device=self.device, dtype=self.dtype)
        Du  = p[:, 0].view(B, 1, 1)
        Dv  = p[:, 1].view(B, 1, 1)
        a   = p[:, 2].view(B, 1)
        b   = p[:, 3].view(B, 1)
        tau = p[:, 4].view(B, 1)

        lap_b = self.lap.unsqueeze(0).expand(B, nx, nx)
        I_b   = self.I.unsqueeze(0).expand(B, nx, nx)
        LU_u, piv_u = torch.linalg.lu_factor(I_b - dt * Du * lap_b)
        LU_v, piv_v = torch.linalg.lu_factor(I_b - dt * Dv * lap_b)

        u = u0.clone().to(self.device, self.dtype)
        v = v0.clone().to(self.device, self.dtype)
        u_hist, v_hist, times = [u.clone()], [v.clone()], [0.0]

        for step in range(n_steps):
            reaction_u = u - u**3 / 3.0 - v
            if I_ext is not None:
                reaction_u = reaction_u + I_ext
            reaction_v = (u + a - b * v) / tau
            rhs_u = (u + dt * reaction_u).unsqueeze(-1)
            rhs_v = (v + dt * reaction_v).unsqueeze(-1)
            u = torch.linalg.lu_solve(LU_u, piv_u, rhs_u).squeeze(-1)
            v = torch.linalg.lu_solve(LU_v, piv_v, rhs_v).squeeze(-1)
            if (step + 1) % save_interval == 0:
                u_hist.append(u.clone()); v_hist.append(v.clone())
                times.append((step + 1) * dt)

        return (torch.stack(u_hist, 1),
                torch.stack(v_hist, 1),
                torch.tensor(times, device=self.device, dtype=self.dtype))

solver = FHNSolverGPU()
print("solver ready; laplacian", tuple(solver.lap.shape))


## Data generation

Each sample draws lambda, builds the IC as the rest state plus a Gaussian bump on u, and integrates to T. The amplitude spans sub- to super-threshold, so the training set holds both failed excitations and full action potentials. That is what lets the operator learn the threshold instead of one fixed outcome.

In [ ]:
def sample_lambda(n, rng):
    lam = np.stack([
        rng.uniform(*DU_RANGE,  n),
        rng.uniform(*DV_RANGE,  n),
        rng.uniform(*A_RANGE,   n),
        rng.uniform(*B_RANGE,   n),
        rng.uniform(*TAU_RANGE, n),
    ], axis=1)
    # resample any non-excitable draws
    for i in range(n):
        while not is_excitable(lam[i, 2], lam[i, 3], lam[i, 4])[0]:
            lam[i, 2] = rng.uniform(*A_RANGE)
            lam[i, 3] = rng.uniform(*B_RANGE)
            lam[i, 4] = rng.uniform(*TAU_RANGE)
    return lam.astype(np.float32)

def make_ics(lam, rng):
    """Return u0,v0 (n,nx) for the given lambda batch with random bumps."""
    n = lam.shape[0]
    x = np.linspace(0.0, L, NX, endpoint=False)
    u0 = np.empty((n, NX), np.float32)
    v0 = np.empty((n, NX), np.float32)
    for i in range(n):
        a, b = lam[i, 2], lam[i, 3]
        us, vs = rest_state(a, b)
        A     = rng.uniform(*AMP_RANGE)
        sigma = rng.uniform(*SIGMA_RANGE)
        x0    = rng.uniform(*X0_RANGE)
        bump  = A * np.exp(-((x - x0) ** 2) / (2 * sigma ** 2))
        u0[i] = us + bump
        v0[i] = vs
    return u0, v0

@torch.no_grad()
def generate(n_samples, seed):
    rng = np.random.default_rng(seed)
    lam = sample_lambda(n_samples, rng)
    u0, v0 = make_ics(lam, rng)
    U, V = [], []
    for s in range(0, n_samples, GEN_CHUNK):
        e = min(s + GEN_CHUNK, n_samples)
        u_b = torch.from_numpy(u0[s:e]).to(device)
        v_b = torch.from_numpy(v0[s:e]).to(device)
        p_b = torch.from_numpy(lam[s:e]).to(device)
        uo, vo, t = solver.solve_batched(u_b, v_b, p_b)
        U.append(uo.cpu()); V.append(vo.cpu())
        print(f"  generated {e}/{n_samples}", end="\r")
    print()
    U = torch.cat(U, 0); V = torch.cat(V, 0)              # (n, n_save+1, nx)
    return U, V, torch.from_numpy(lam), t.cpu()

t0 = time.time()
print("generating train set...")
Utr, Vtr, Ltr, tgrid = generate(N_TRAIN, seed=SEED)
print("generating val set...")
Uva, Vva, Lva, _     = generate(N_VAL, seed=SEED + 999)
print(f"done in {time.time()-t0:.1f}s")
print("train traj:", tuple(Utr.shape), " val traj:", tuple(Uva.shape))


### Sanity check: space-time plots

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for j in range(4):
    u = Utr[j].numpy()             # (n_save+1, nx)
    v = Vtr[j].numpy()
    im0 = axes[0, j].imshow(u, aspect="auto", origin="lower",
                            extent=[0, L, 0, T], cmap="inferno")
    im1 = axes[1, j].imshow(v, aspect="auto", origin="lower",
                            extent=[0, L, 0, T], cmap="viridis")
    lam = Ltr[j].numpy()
    axes[0, j].set_title(f"Du={lam[0]:.3f} tau={lam[4]:.1f}", fontsize=9)
    plt.colorbar(im0, ax=axes[0, j], fraction=0.046)
    plt.colorbar(im1, ax=axes[1, j], fraction=0.046)
axes[0, 0].set_ylabel("u  (time ->)")
axes[1, 0].set_ylabel("v  (time ->)")
for ax in axes[1]:
    ax.set_xlabel("x")
plt.suptitle("Excitable FHN: travelling action potentials (u) and recovery (v)")
plt.tight_layout(); plt.savefig("excitable_spacetime.png", dpi=130); plt.show()


## Model: FiLM-conditioned parametric FNO

Same architecture as the main paper's fno.py (1D path): spectral conv, 1x1 conv, GELU, InstanceNorm, and a learnable residual per layer, with per-layer FiLM modulation from a parameter encoder and a small global input skip.

In [ ]:
class SpectralConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, modes):
        super().__init__()
        self.in_ch, self.out_ch, self.modes = in_ch, out_ch, modes
        scale = 1.0 / (in_ch * out_ch)
        self.weights = nn.Parameter(scale * torch.randn(in_ch, out_ch, modes, dtype=torch.cfloat))

    def forward(self, x):
        B, _, nx = x.shape
        x_ft = torch.fft.rfft(x, dim=-1)
        out_ft = torch.zeros(B, self.out_ch, nx // 2 + 1, dtype=torch.cfloat, device=x.device)
        out_ft[:, :, :self.modes] = torch.einsum(
            "bix,iox->box", x_ft[:, :, :self.modes], self.weights)
        return torch.fft.irfft(out_ft, n=nx, dim=-1)


class FourierLayer(nn.Module):
    def __init__(self, width, modes):
        super().__init__()
        self.spectral = SpectralConv1d(width, width, modes)
        self.w = nn.Conv1d(width, width, 1)
        self.norm = nn.InstanceNorm1d(width, affine=True)
        self.residual_weight = nn.Parameter(torch.ones(1))

    def forward(self, x):
        residual = x
        out = F.gelu(self.spectral(x) + self.w(x))
        out = self.norm(out)
        return out + self.residual_weight * residual


class ParameterEncoder(nn.Module):
    def __init__(self, n_params, width, hidden=None):
        super().__init__()
        hidden = hidden or width * 2
        self.encoder = nn.Sequential(
            nn.Linear(n_params, hidden), nn.GELU(), nn.LayerNorm(hidden),
            nn.Linear(hidden, width), nn.GELU(),
            nn.Linear(width, width), nn.LayerNorm(width))

    def forward(self, p):
        return self.encoder(p)


class ParametricFNO(nn.Module):
    def __init__(self, modes=MODES, width=WIDTH, n_layers=N_LAYERS,
                 in_ch=IN_CH, out_ch=OUT_CH, n_params=N_PARAMS):
        super().__init__()
        self.param_encoder = ParameterEncoder(n_params, width)
        self.film_gamma = nn.ModuleList([nn.Linear(width, width) for _ in range(n_layers)])
        self.film_beta  = nn.ModuleList([nn.Linear(width, width) for _ in range(n_layers)])
        self.lift = nn.Sequential(nn.Conv1d(in_ch, width * 2, 1), nn.GELU(),
                                  nn.Conv1d(width * 2, width, 1))
        self.layers = nn.ModuleList([FourierLayer(width, modes) for _ in range(n_layers)])
        self.projection = nn.Sequential(nn.Conv1d(width, width, 1), nn.GELU(),
                                        nn.Conv1d(width, out_ch, 1))
        self.global_residual = nn.Parameter(torch.ones(1) * 0.1)

    def forward(self, x, params):
        input_x = x
        x = self.lift(x)
        pf = self.param_encoder(params)
        for i, layer in enumerate(self.layers):
            g = self.film_gamma[i](pf).unsqueeze(-1)
            b = self.film_beta[i](pf).unsqueeze(-1)
            x = g * layer(x) + b
        x = self.projection(x)
        return x + self.global_residual * input_x

    @torch.no_grad()
    def rollout(self, x0, n_steps, params):
        traj = [x0]; x = x0
        for _ in range(n_steps):
            x = self.forward(x, params); traj.append(x)
        return torch.stack(traj, dim=1)

model = ParametricFNO().to(device)
print("params:", sum(p.numel() for p in model.parameters()))


## Normalization and single-step training

Learn the one-step map on consecutive saved frames. Inputs, outputs, and lambda are z-scored. Loss is relative L2 summed over the two channels.

In [ ]:
def build_pairs(U, V):
    """(N,Tn,nx),(N,Tn,nx) -> X,Y:(N*(Tn-1),2,nx), Pidx mapping to sample."""
    N, Tn, nx = U.shape
    X_in  = torch.stack([U[:, :-1], V[:, :-1]], dim=2)   # (N,Tn-1,2,nx)
    X_out = torch.stack([U[:, 1:],  V[:, 1:]],  dim=2)
    X_in  = X_in.reshape(-1, 2, nx)
    X_out = X_out.reshape(-1, 2, nx)
    pidx  = torch.arange(N).view(N, 1).expand(N, Tn - 1).reshape(-1)
    return X_in, X_out, pidx

Xtr, Ytr, Ptr = build_pairs(Utr, Vtr)
Xva, Yva, Pva = build_pairs(Uva, Vva)
print("train pairs:", Xtr.shape[0], " val pairs:", Xva.shape[0])

# normalization stats from train inputs
um, us = Xtr[:, 0].mean(), Xtr[:, 0].std()
vm, vs = Xtr[:, 1].mean(), Xtr[:, 1].std()
pm, ps = Ltr.mean(0), Ltr.std(0)
chan_m = torch.tensor([um, vm]).view(1, 2, 1)
chan_s = torch.tensor([us, vs]).view(1, 2, 1)
print("u  mean/std", float(um), float(us))
print("v  mean/std", float(vm), float(vs))

def norm_x(x):  return (x - chan_m) / chan_s
def denorm(x):  return x * chan_s.to(x.device) + chan_m.to(x.device)
def norm_p(p):  return (p - pm) / ps


In [ ]:
def rel_l2(pred, tgt):
    """per-sample relative L2 averaged over batch and channels."""
    num = torch.linalg.vector_norm(pred - tgt, dim=-1)
    den = torch.linalg.vector_norm(tgt, dim=-1) + 1e-8
    return (num / den).mean()

# normalize once
Xtr_n = norm_x(Xtr); Ytr_n = norm_x(Ytr)
Xva_n = norm_x(Xva); Yva_n = norm_x(Yva)
Ltr_n = norm_p(Ltr); Lva_n = norm_p(Lva)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
n_pairs = Xtr_n.shape[0]

for epoch in range(EPOCHS):
    model.train()
    perm = torch.randperm(n_pairs)
    running = 0.0
    for s in range(0, n_pairs, BATCH):
        idx = perm[s:s + BATCH]
        x = Xtr_n[idx].to(device)
        y = Ytr_n[idx].to(device)
        p = Ltr_n[Ptr[idx]].to(device)
        opt.zero_grad()
        pred = model(x, p)
        loss = rel_l2(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step()
        running += loss.item() * idx.numel()
    sched.step()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        model.eval()
        with torch.no_grad():
            vs_ = 0.0
            for s in range(0, Xva_n.shape[0], 512):
                x = Xva_n[s:s+512].to(device)
                y = Yva_n[s:s+512].to(device)
                p = Lva_n[Pva[s:s+512]].to(device)
                vs_ += rel_l2(model(x, p), y).item() * x.shape[0]
            vs_ /= Xva_n.shape[0]
        print(f"epoch {epoch+1:3d}  train relL2 {running/n_pairs:.4e}  val relL2 {vs_:.4e}")


## Single-step metrics

In [ ]:
@torch.no_grad()
def single_step_metrics(Xn, Yn, P, pidx):
    model.eval()
    preds, tgts = [], []
    for s in range(0, Xn.shape[0], 512):
        x = Xn[s:s+512].to(device)
        p = P[pidx[s:s+512]].to(device)
        preds.append(denorm(model(x, p)).cpu())
        tgts.append(denorm(Yn[s:s+512]))
    pred = torch.cat(preds); tgt = torch.cat(tgts)
    out = {}
    for c, name in enumerate(["u", "v"]):
        pc, tc = pred[:, c], tgt[:, c]
        num = torch.linalg.vector_norm(pc - tc, dim=-1)
        den = torch.linalg.vector_norm(tc, dim=-1) + 1e-8
        rl2 = (num / den).mean().item()
        mse = ((pc - tc) ** 2).mean().item()
        mae = (pc - tc).abs().mean().item()
        mxa = (pc - tc).abs().max().item()
        ss_res = ((tc - pc) ** 2).sum().item()
        ss_tot = ((tc - tc.mean()) ** 2).sum().item()
        r2 = 1.0 - ss_res / ss_tot
        out[name] = dict(relL2=rl2, MSE=mse, MAE=mae, MaxAE=mxa, R2=r2)
    return out

m = single_step_metrics(Xva_n, Yva_n, Lva_n, Pva)
print(f"{'metric':>8} | {'u':>12} | {'v':>12}")
for k in ["relL2", "MSE", "MAE", "MaxAE", "R2"]:
    print(f"{k:>8} | {m['u'][k]:>12.4e} | {m['v'][k]:>12.4e}")


## Long autoregressive rollout

In [ ]:
@torch.no_grad()
def rollout_eval(U, V, lam, n_steps=N_SAVE):   # lam = params, not the domain length L
    model.eval()
    x0 = torch.stack([U[:, 0], V[:, 0]], dim=1)            # (N,2,nx)
    x0n = norm_x(x0).to(device)
    p   = norm_p(lam).to(device)
    traj = model.rollout(x0n, n_steps, p)                  # (N,n_steps+1,2,nx) normalized
    traj = denorm(traj)
    gt = torch.stack([U, V], dim=2).to(traj.device)        # (N,Tn,2,nx)
    errs = []
    for c in range(2):
        num = torch.linalg.vector_norm(traj[:, :, c] - gt[:, :, c], dim=-1)
        den = torch.linalg.vector_norm(gt[:, :, c], dim=-1) + 1e-8
        errs.append((num / den).mean(0).cpu().numpy())     # (Tn,)
    return traj.cpu(), np.stack(errs)                      # errs:(2,Tn)

traj_pred, roll_err = rollout_eval(Uva, Vva, Lva)
print(f"u  rollout relL2  step1={roll_err[0,1]:.4e}  step50={roll_err[0,50]:.4e}  step100={roll_err[0,100]:.4e}")
print(f"v  rollout relL2  step1={roll_err[1,1]:.4e}  step50={roll_err[1,50]:.4e}  step100={roll_err[1,100]:.4e}")

steps = np.arange(roll_err.shape[1]) * DT_SAVE
plt.figure(figsize=(7, 4))
plt.plot(steps, roll_err[0], label="u", lw=2)
plt.plot(steps, roll_err[1], label="v", lw=2)
plt.xlabel("time"); plt.ylabel("relative L2"); plt.legend()
plt.title("Autoregressive rollout error (excitable FHN)")
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("excitable_rollout_error.png", dpi=130); plt.show()


### Live travelling action potential

Aggregate error is small, but the real test is the propagating pulse. We pick the validation sample whose excitation sweeps the widest stretch of space (dropping any rare domain-filling case) and check that the FNO rollout reproduces the full space-time pulse for both u and v.

In [ ]:
# pick a clearly propagating sample: widest excited front, dropping any domain-filling case
fire   = (Uva.numpy() > 0.5)               # supra-threshold mask
ever   = fire.any(axis=1)                  # cells excited at any time
extent = ever.sum(axis=1)                  # spatial reach
valid  = extent < 0.9 * NX                 # exclude domain-filling
j = int(np.where(valid, extent, -1).argmax())
print(f"selected sample j={j}  params(Du,Dv,a,b,tau)={np.round(Lva[j].numpy(),4)}")
print(f"  u_peak={Uva[j].numpy().max():.3f}  spatial reach={extent[j]}/{NX} cells")

u_true = Uva[j].numpy(); u_pred = traj_pred[j, :, 0].numpy()
v_true = Vva[j].numpy(); v_pred = traj_pred[j, :, 1].numpy()

fig, ax = plt.subplots(2, 3, figsize=(15, 8))
for row, (gt, pr, name, cmap) in enumerate(
        [(u_true, u_pred, "u", "inferno"), (v_true, v_pred, "v", "viridis")]):
    vmin, vmax = gt.min(), gt.max()
    ax[row,0].imshow(gt, aspect="auto", origin="lower", extent=[0,L,0,T],
                     vmin=vmin, vmax=vmax, cmap=cmap)
    ax[row,0].set_title(f"{name}  ground truth")
    ax[row,1].imshow(pr, aspect="auto", origin="lower", extent=[0,L,0,T],
                     vmin=vmin, vmax=vmax, cmap=cmap)
    ax[row,1].set_title(f"{name}  FNO rollout")
    im = ax[row,2].imshow(np.abs(gt - pr), aspect="auto", origin="lower",
                          extent=[0,L,0,T], cmap="magma")
    ax[row,2].set_title(f"{name}  |error|"); plt.colorbar(im, ax=ax[row,2], fraction=0.046)
    ax[row,0].set_ylabel("time")
for a_ in ax[-1]: a_.set_xlabel("x")
fig.suptitle(f"Travelling action potential — solver vs FNO rollout (sample j={j})", y=1.01)
plt.tight_layout(); plt.savefig("excitable_rollout_field.png", dpi=130); plt.show()


In [ ]:
# snapshots of the moving pulse, FNO over ground truth
xg = np.linspace(0.0, L, NX, endpoint=False)
Tn = u_true.shape[0]
snap = np.linspace(0, Tn - 1, 6).astype(int)
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True, sharey=True)
for a_, k in zip(axes.ravel(), snap):
    a_.plot(xg, u_true[k], color="k",       lw=2.4, label="solver")
    a_.plot(xg, u_pred[k], color="crimson",  lw=1.9, ls="--", label="FNO")
    a_.set_title(f"t = {k*DT_SAVE:.1f}"); a_.grid(alpha=0.3)
axes[0,0].legend(loc="upper right")
for a_ in axes[-1]: a_.set_xlabel("x")
for a_ in axes[:,0]: a_.set_ylabel("u")
fig.suptitle(f"Live action-potential replication — u(x) snapshots (sample j={j})", y=1.01)
plt.tight_layout(); plt.savefig("excitable_rollout_snapshots.png", dpi=130); plt.show()


In [ ]:
# animation: FNO (dashed) tracks the solver (solid)
import matplotlib.animation as animation
from IPython.display import HTML

lo = min(u_true.min(), u_pred.min()) - 0.2
hi = max(u_true.max(), u_pred.max()) + 0.2
figA, axA = plt.subplots(figsize=(7.5, 4.2))
axA.set_xlim(0, L); axA.set_ylim(lo, hi)
axA.set_xlabel("x"); axA.set_ylabel("u"); axA.grid(alpha=0.3)
(l_gt,) = axA.plot([], [], color="k",      lw=2.6, label="solver")
(l_fn,) = axA.plot([], [], color="crimson", lw=2.0, ls="--", label="FNO")
ttl = axA.set_title(""); axA.legend(loc="upper right")

def _init():
    l_gt.set_data([], []); l_fn.set_data([], []); return l_gt, l_fn, ttl

def _frame(k):
    l_gt.set_data(xg, u_true[k]); l_fn.set_data(xg, u_pred[k])
    ttl.set_text(f"travelling action potential   t = {k*DT_SAVE:5.2f}")
    return l_gt, l_fn, ttl

ani = animation.FuncAnimation(figA, _frame, frames=Tn, init_func=_init,
                              interval=80, blit=True)
plt.close(figA)
HTML(ani.to_jshtml())


## All-or-none firing

Fix one parameter set and sweep the bump amplitude A. For each A we take the peak of u in a window after the stimulus. Below threshold it decays to rest; above threshold a full action potential fires. We overlay solver and FNO to show the operator captures the jump, not a smooth interpolation.

In [ ]:
# mid-box excitable parameters
lam_fix = np.array([0.015, 0.002, 0.70, 0.80, 12.0], dtype=np.float32)
us_fix, vs_fix = rest_state(lam_fix[2], lam_fix[3])
print(f"rest state: u*={us_fix:.3f} v*={vs_fix:.3f}")

amps = np.linspace(0.05, 2.5, 29).astype(np.float32)
x = np.linspace(0.0, L, NX, endpoint=False)
sigma_fix, x0_fix = 0.4, L / 2

# one IC per amplitude
u0 = np.stack([us_fix + A * np.exp(-((x - x0_fix) ** 2) / (2 * sigma_fix ** 2)) for A in amps]).astype(np.float32)
v0 = np.full_like(u0, vs_fix)
lam_b = np.tile(lam_fix, (len(amps), 1))

# ground-truth solver
ug, vg, _ = solver.solve_batched(torch.from_numpy(u0).to(device),
                                 torch.from_numpy(v0).to(device),
                                 torch.from_numpy(lam_b).to(device))
ug = ug.cpu()                                              # (A, n_save+1, nx)

# FNO rollout from the same ICs
x0n = norm_x(torch.stack([torch.from_numpy(u0), torch.from_numpy(v0)], dim=1)).to(device)
pn  = norm_p(torch.from_numpy(lam_b)).to(device)
with torch.no_grad():
    tr = denorm(model.rollout(x0n, N_SAVE, pn)).cpu()      # (A, n_save+1, 2, nx)
uf = tr[:, :, 0]

# peak u in a window after the bump but before the fronts reach the far side.
# t=0 would just echo the amplitude; too late catches the recovered state.
w0, w1 = int(2.0 / DT_SAVE), int(12.0 / DT_SAVE)
resp_true = ug[:, w0:w1].amax(dim=(1, 2)).numpy()
resp_fno  = uf[:, w0:w1].amax(dim=(1, 2)).numpy()

plt.figure(figsize=(7, 4.5))
plt.axhline(us_fix, color="gray", ls=":", label="rest $u^*$")
plt.plot(amps, resp_true, "o-", label="solver (ground truth)", lw=2)
plt.plot(amps, resp_fno,  "s--", label="FNO rollout", lw=2)
plt.xlabel("stimulus amplitude  A")
plt.ylabel(f"peak $u$  (t in [{w0*DT_SAVE:.0f},{w1*DT_SAVE:.0f}])")
plt.title("All-or-none firing: sharp excitation threshold")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("excitable_all_or_none.png", dpi=140); plt.show()


## Conduction velocity vs diffusion

Fire a single pulse and track how far the excited region spreads; the slope is the conduction velocity c. Theory says c scales as sqrt(Du). We check that the solver and the FNO agree.

In [ ]:
def front_distance(u_frame, x, x0):
    # farthest the excited region (u>0) has spread from x0 on the ring
    idx = np.where(u_frame > 0.0)[0]
    if idx.size == 0:
        return np.nan
    return float(np.max(np.abs(((x[idx] - x0 + L / 2) % L) - L / 2)))

def conduction_velocity(u_traj, x, t, x0):
    d = np.array([front_distance(u_traj[k], x, x0) for k in range(u_traj.shape[0])])
    # fit while the front is propagating, before collision
    good = np.isfinite(d) & (d > 0.3) & (d < 0.42 * L)
    if good.sum() < 4:
        return np.nan, d
    c = np.polyfit(t[good], d[good], 1)[0]
    return c, d

Du_test = np.array([0.006, 0.010, 0.015, 0.020, 0.025, 0.030], dtype=np.float32)
x = np.linspace(0.0, L, NX, endpoint=False)
tt = (np.arange(N_SAVE + 1) * DT_SAVE)
x0_cv = L / 2

u0 = np.tile(us_fix + 2.5 * np.exp(-((x - x0_cv) ** 2) / (2 * 0.4 ** 2)), (len(Du_test), 1)).astype(np.float32)
v0 = np.full_like(u0, vs_fix)
lam_b = np.tile(lam_fix, (len(Du_test), 1)); lam_b[:, 0] = Du_test

ug, _, _ = solver.solve_batched(torch.from_numpy(u0).to(device),
                                torch.from_numpy(v0).to(device),
                                torch.from_numpy(lam_b).to(device))
ug = ug.cpu().numpy()

x0n = norm_x(torch.stack([torch.from_numpy(u0), torch.from_numpy(v0)], dim=1)).to(device)
pn  = norm_p(torch.from_numpy(lam_b)).to(device)
with torch.no_grad():
    uf = denorm(model.rollout(x0n, N_SAVE, pn)).cpu().numpy()[:, :, 0]

c_true = [conduction_velocity(ug[i], x, tt, x0_cv)[0] for i in range(len(Du_test))]
c_fno  = [conduction_velocity(uf[i], x, tt, x0_cv)[0] for i in range(len(Du_test))]
for D, ct, cf in zip(Du_test, c_true, c_fno):
    print(f"Du={D:.3f}  c_solver={ct:.4f}  c_FNO={cf:.4f}")

plt.figure(figsize=(6.5, 4.5))
plt.plot(np.sqrt(Du_test), c_true, "o-", label="solver", lw=2)
plt.plot(np.sqrt(Du_test), c_fno,  "s--", label="FNO", lw=2)
plt.xlabel(r"$\sqrt{D_u}$"); plt.ylabel("conduction velocity  c")
plt.title(r"Conduction velocity $\propto \sqrt{D_u}$")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("excitable_conduction_velocity.png", dpi=140); plt.show()


## Save artifacts

Saves the checkpoint (with normalization stats) and a metrics JSON so the figures and tables can be reproduced.

In [ ]:
ckpt = {
    "model_state": model.state_dict(),
    "norm": {"um": float(um), "us": float(us), "vm": float(vm), "vs": float(vs),
             "pm": pm.tolist(), "ps": ps.tolist()},
    "config": {"NX": NX, "DT": DT, "T": T, "N_SAVE": N_SAVE, "DT_SAVE": DT_SAVE,
               "MODES": MODES, "WIDTH": WIDTH, "N_LAYERS": N_LAYERS,
               "DU_RANGE": DU_RANGE, "DV_RANGE": DV_RANGE, "A_RANGE": A_RANGE,
               "B_RANGE": B_RANGE, "TAU_RANGE": TAU_RANGE, "AMP_RANGE": AMP_RANGE},
}
torch.save(ckpt, "excitable_fno_s42.pt")

results = {
    "single_step": m,
    "rollout_relL2_u": roll_err[0].tolist(),
    "rollout_relL2_v": roll_err[1].tolist(),
    "conduction": {"Du": Du_test.tolist(), "c_solver": c_true, "c_fno": c_fno},
    "threshold": {"amps": amps.tolist(), "resp_solver": resp_true.tolist(),
                  "resp_fno": resp_fno.tolist()},
}
with open("excitable_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("saved: excitable_fno_s42.pt, excitable_results.json")
print("figures: excitable_spacetime.png, excitable_rollout_error.png,")
print("         excitable_rollout_field.png, excitable_rollout_snapshots.png,")
print("         excitable_all_or_none.png, excitable_conduction_velocity.png")


## Notes and extensions

- Conduction block: add a third input channel for I_ext through both the solver and the FNO lift, and train on S1-S2 protocols. Bump IN_CH to 3 and stack I_ext in build_pairs.
- Refractory period: re-stimulate after a delay and sweep it; the smallest delay that re-fires is the refractory period.
- The single-step and rollout numbers plug into the same table format as the main paper; all-or-none and conduction velocity are the new contributions.